# Pipeline de expresiones faciales equinas

Este notebook **ejecuta los demás en orden**. Es el único que hace falta abrir para correr el
pipeline completo.

Resuelve el problema que tenía el pipeline partido en notebooks sueltos: cada notebook de Colab
arrancaba con un `/content` vacío, así que el `02` no encontraba los clips y los CSV que había
generado el `01`. Ahora todos los artefactos viven en una carpeta de Google Drive y el paso de
datos entre etapas ocurre por ahí.

Qué hace, concretamente:

- **saltea** las etapas cuyos resultados ya están en Drive, para poder avanzar de a tramos a lo
  largo de varias sesiones;
- **corta la cadena** si una etapa falla, en vez de seguir con las que dependen de ella;
- **guarda una copia ejecutada** de cada notebook, con sus salidas y los parámetros usados, como
  evidencia reproducible de qué se corrió y con qué resultado.

Colab corta la sesión por límite de tiempo o por inactividad. Si eso pasa a mitad de camino,
alcanza con volver a ejecutar este notebook: retoma donde había quedado.

## Paso 0 — Arranque: montar Drive, traer el código e instalar papermill

In [ ]:
# ============================================================
# ARRANQUE DEL ORQUESTADOR
# ============================================================
# Autocontenida a propósito: corre antes de que el paquete `equinos` esté disponible.
#
# A diferencia de los notebooks de etapa, éste necesita además los .ipynb de las etapas en
# disco, porque papermill los ejecuta como archivos.
#
# El repo es privado, así que un `git clone` sin credenciales no alcanza. El token va en los
# secrets de Colab (panel izquierdo, icono de llave) y NUNCA dentro del notebook: lo que se
# escribe en una celda queda en el archivo y se comparte con él.

import shutil
import subprocess
import sys
from pathlib import Path

NOMBRE_PROYECTO = "tesina_equinos"
REPO = "github.com/sergioyanez/Tesina-UGR.git"

# True vuelve a traer el código del repo aunque ya esté en Drive. Ponerlo en True después de
# cambiar un notebook o el paquete.
SINCRONIZAR_CODIGO = False

# ============================================================
# CARPETA DE VIDEOS DE PRUEBA
# ============================================================
# La carpeta original está en "Compartidos conmigo".
#
# IMPORTANTE:
# Para que Colab pueda leerla como ruta, agregá un ACCESO DIRECTO a "Mi unidad":
#
#   Drive -> Compartidos conmigo -> Videos de prueba
#   clic derecho -> Organizar -> Agregar acceso directo -> Mi unidad
#
# Eso NO copia ni mueve los videos. Solo crea un acceso directo.
#
# Luego Colab la puede encontrar dentro de:
#   /content/drive/MyDrive/...

NOMBRE_CARPETA_VIDEOS_PRUEBA = "Videos de prueba"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False


def buscar_carpeta_por_nombre(base: Path, nombre_carpeta: str) -> Path:
    """
    Busca una carpeta por nombre dentro de una ruta base.
    Sirve para encontrar accesos directos de Drive montados en Colab.
    """

    nombre_buscado = nombre_carpeta.lower().strip()

    # Primero prueba la ubicación más común:
    candidata_directa = base / nombre_carpeta
    if candidata_directa.exists() and candidata_directa.is_dir():
        return candidata_directa

    # Después busca dentro de Mi unidad.
    for ruta in base.rglob("*"):
        if ruta.is_dir() and ruta.name.lower().strip() == nombre_buscado:
            return ruta

    raise FileNotFoundError(
        f"No se encontró la carpeta '{nombre_carpeta}' dentro de {base}"
    )


if EN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)

    PROYECTO = Path("/content/drive/MyDrive/tesina_equinos")
    CODIGO = PROYECTO / "codigo"

    CODIGO.mkdir(parents=True, exist_ok=True)

    print(f"Proyecto: {PROYECTO}")
    print(f"Código:   {CODIGO}")

    # ========================================================
    # BUSCAR "VIDEOS DE PRUEBA" EN DRIVE MONTADO
    # ========================================================
    # Esta carpeta debe estar como ACCESO DIRECTO en Mi unidad.
    # No se usa gdown.
    # No se usa API de Drive.
    # No se busca en "Compartidos conmigo" porque Colab no monta esa sección como carpeta.

    MI_UNIDAD = Path("/content/drive/MyDrive")

    try:
        VIDEOS_PRUEBA = buscar_carpeta_por_nombre(
            base=MI_UNIDAD,
            nombre_carpeta=NOMBRE_CARPETA_VIDEOS_PRUEBA,
        )
    except FileNotFoundError:
        raise SystemExit(
            f"No encontré la carpeta '{NOMBRE_CARPETA_VIDEOS_PRUEBA}' dentro de Mi unidad.\n\n"
            "Tu carpeta está en 'Compartidos conmigo', pero Colab no puede leer directamente\n"
            "esa sección como una ruta normal.\n\n"
            "Solución obligatoria para este método:\n"
            "  1. Abrí Google Drive.\n"
            "  2. Entrá a 'Compartidos conmigo'.\n"
            "  3. Clic derecho sobre 'Videos de prueba'.\n"
            "  4. Elegí 'Organizar' o 'Agregar acceso directo'.\n"
            "  5. Guardalo en 'Mi unidad'.\n"
            "  6. Volvé a ejecutar esta celda.\n\n"
            "Eso NO copia ni mueve los videos. Solo crea un acceso directo."
        )

    print(f"Videos de prueba encontrados en: {VIDEOS_PRUEBA}")

    extensiones_video = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

    videos_detectados = [
        ruta for ruta in VIDEOS_PRUEBA.rglob("*")
        if ruta.suffix.lower() in extensiones_video
    ]

    print(f"Cantidad de videos detectados: {len(videos_detectados)}")

    for video in videos_detectados:
        print(f"  - {video}")

    if len(videos_detectados) == 0:
        raise SystemExit(
            f"La carpeta '{VIDEOS_PRUEBA}' existe, pero no encontré videos.\n"
            "Extensiones buscadas: .mp4, .avi, .mov, .mkv, .webm"
        )

    # ========================================================
    # SINCRONIZACIÓN DEL CÓDIGO DESDE GITHUB
    # ========================================================

    if SINCRONIZAR_CODIGO or not (CODIGO / "equinos").is_dir():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception as error:
            # Se distingue "el secret no existe" de "existe pero este notebook no tiene acceso".
            # El interruptor por notebook es lo que más se olvida, y RE-SUBIR este archivo a
            # Drive lo apaga: para Colab es un notebook nuevo y el permiso arranca en off.
            if type(error).__name__ == "NotebookAccessError":
                raise SystemExit(
                    "El secret GITHUB_TOKEN existe pero este notebook no tiene acceso.\n"
                    "  Panel izquierdo -> icono de llave -> activá el interruptor de este notebook.\n"
                    "  Si acabás de re-subir 00_pipeline.ipynb, es esto: el permiso no se hereda."
                )
            token = None
            fallo_token = f"{type(error).__name__}: {error}"
        else:
            fallo_token = "userdata.get('GITHUB_TOKEN') devolvió vacío"

        if not token:
            # Antes este mensaje decía "Falta el código", que es engañoso cuando el código está
            # y lo que falla es el token: manda a resolver el problema equivocado.
            hay_codigo = (CODIGO / "equinos").is_dir()
            estado = ("El código YA está en Drive; lo que falta es el token para actualizarlo.\n"
                      "  Si no lo necesitás ahora, poné SINCRONIZAR_CODIGO = False y seguí\n"
                      "  --pero entonces corrés la versión que ya está, no la del repo--."
                      if hay_codigo else f"Además falta el código en {CODIGO}.")
            raise SystemExit(
                f"No se pudo leer el secret GITHUB_TOKEN ({fallo_token}).\n"
                f"  {estado}\n"
                "  Opción A: panel izquierdo -> icono de llave -> crear GITHUB_TOKEN y activar\n"
                "            el interruptor PARA ESTE NOTEBOOK.\n"
                f"  Opción B: copiá a mano `equinos/` y los notebooks NN_*.ipynb dentro de {CODIGO}."
            )

        clon = Path("/content/_repo_tesina")
        shutil.rmtree(clon, ignore_errors=True)

        resultado = subprocess.run(
            ["git", "clone", "--depth", "1", f"https://{token}@{REPO}", str(clon)],
            capture_output=True,
            text=True,
        )

        if resultado.returncode != 0:
            # No se imprime el comando ni stderr: contienen el token.
            raise SystemExit(
                "Falló el clonado. Verificá que GITHUB_TOKEN tenga permiso de lectura del repo."
            )

        shutil.copytree(clon / "equinos", CODIGO / "equinos", dirs_exist_ok=True)

        # Este mismo notebook queda EXCLUIDO de la copia. Colab lo tiene abierto desde Drive, y
        # al autoguardar escribiría su versión en memoria encima de la recién sincronizada:
        # el sync se revertiría sin avisar y la corrida siguiente usaría el código viejo.
        ORQUESTADOR = "00_pipeline.ipynb"

        copiados = []

        for notebook in sorted(clon.glob("[0-9][0-9]_*.ipynb")):
            if notebook.name == ORQUESTADOR:
                continue

            shutil.copy(notebook, CODIGO / notebook.name)
            copiados.append(notebook.name)

        # Se deja la huella del commit clonado. Sin esto, "qué código corrió" es indecidible
        # después: el 18/08 una corrida de 62 min ejecutó el notebook viejo porque los commits
        # no estaban pusheados, y el síntoma recién apareció una hora más tarde como un
        # artefacto faltante.
        sha = subprocess.run(["git", "-C", str(clon), "rev-parse", "--short", "HEAD"],
                             capture_output=True, text=True).stdout.strip()
        (CODIGO / "VERSION_CODIGO.txt").write_text(sha, encoding="utf-8")

        print(f"Código sincronizado en {CODIGO}")
        print("  equinos/ + " + ", ".join(copiados))
        print(f"  {ORQUESTADOR} no se toca: es el que estás ejecutando.")
        print("  Si cambió, copialo a mano a esa carpeta y reabrilo desde Drive.")

    sys.path.insert(0, str(CODIGO))

    # Se imprime SIEMPRE, se haya sincronizado o no: el caso peligroso es justamente el de
    # SINCRONIZAR_CODIGO = False, donde no hay ninguna otra señal de qué versión hay en Drive.
    version = CODIGO / "VERSION_CODIGO.txt"
    if version.exists():
        print(f"\nVersión del código en Drive: {version.read_text().strip()}")
    else:
        print("\nVersión del código en Drive: desconocida (copiado a mano o sync anterior a "
              "esta salvaguarda).")
    print("Comparala con el último commit del repo antes de lanzar una etapa larga.")

else:
    CODIGO = Path.cwd()
    VIDEOS_PRUEBA = CODIGO / "Videos de prueba"
    sys.path.insert(0, str(CODIGO))


# ============================================================
# DEPENDENCIAS DE TODO EL PIPELINE
# ============================================================
# Se instalan acá, de una sola vez, ANTES de lanzar ninguna etapa.
#
# Los notebooks de etapa corren en kernels separados pero comparten el site-packages de la
# sesión. Si cada uno instala lo suyo sobre la marcha, el segundo le cambia las versiones al
# primero por debajo. En particular DeepLabCut fija numpy<2, y Colab trae scipy y pandas
# compilados contra numpy 2: instalarlo tarde rompe el entorno entero.
#
# Instalando el conjunto completo antes de arrancar, cada kernel de etapa se encuentra un disco
# ya consistente. Este kernel, el del orquestador, no importa numpy ni pandas justamente para
# no quedar atado a esas versiones.

!pip install -q papermill deeplabcut "scipy<1.14" ultralytics opencv-python pandas numpy tqdm matplotlib

from equinos import cache, checkpoint, etapas, rutas

RAIZ = rutas.inicializar()

print(f"RAIZ del proyecto: {RAIZ}")
print(f"Ruta de videos de prueba: {VIDEOS_PRUEBA}")

## Paso 1 — Parámetros compartidos

In [ ]:
# ============================================================
# PARÁMETROS DEL PIPELINE
# ============================================================
# Un solo lugar para los valores que atraviesan varias etapas. Papermill los inyecta en cada
# notebook, así que barrer un umbral no requiere abrir ni editar ninguna celda de las etapas.
#
# Sólo hay que listar lo que se quiera cambiar: lo que no aparezca acá toma el valor por defecto
# declarado en la celda `parameters` del notebook correspondiente.

PARAMETROS = {
    "preparar_dataset": {
        # "DURACION_CLIP": 6.0,
        # "NUM_FRAMES": 16,
    },
    "roi_normalizacion": {
        # "CONF_MINIMA_DETECCION": 0.50,
        # "MARGEN_ROI": 0.45,
        # "COBERTURA_MINIMA_ACEPTABLE": 0.70,
        #
        # Keypoint del hocico. Sin fijar, se usa el primer candidato que exponga el modelo, que
        # es el orden de una lista y no un criterio. Se elige con `pose.barrido()`, que compara
        # los candidatos sobre la pose ya inferida y no obliga a reinferir.
        # "BODYPART_HOCICO": "upper_jaw",
    },
}

# Etapas a re-ejecutar aunque ya estén completas. Ojo: descarta sus artefactos.
ETAPAS_A_FORZAR = []

## Paso 2 — Diagnóstico: en qué estado está el pipeline

No ejecuta ni modifica nada. Conviene mirarlo al abrir una sesión nueva, para saber qué falta.

In [ ]:
reporte = etapas.diagnostico()

## Paso 3 — Ejecutar

Las etapas completas se saltean. Si una falla, el pipeline se detiene ahí: seguir con las que
dependen de ella sólo produciría un segundo error más confuso que el primero.

In [ ]:
import shutil
import threading
import time
from datetime import datetime

import papermill as pm

# ============================================================
# EJECUCIÓN ENCADENADA
# ============================================================
# Cada etapa corre en su propio kernel. Las etapas NO se comunican por memoria compartida: el
# único canal entre ellas son los artefactos persistidos en la raíz de proyecto. Es lo que
# permite que una sesión ejecute el 01, se corte, y otra sesión al día siguiente ejecute el 02.

LATIDO_SEGUNDOS = 60


def ejecutar_etapa(nombre, entrada, salida, parametros, inicio):
    """Ejecuta un notebook mostrando señales de vida, y devuelve el fallo o None.

    Papermill bloquea hasta terminar, y ninguna de sus dos opciones sirve tal cual:
    con log_output=True vuelca la salida entera del hijo —las barras de 92 clips más lo que
    imprime DeepLabCut por video— y llega a colgar la pestaña de Colab; con log_output=False
    no se ve nada durante una hora y es indistinguible de un cuelgue.

    El punto medio: ejecutar en un hilo y leer el checkpoint desde acá una vez por minuto. Es
    salida mínima y progreso real, medido sobre lo que quedó persistido en Drive.
    """
    resultado = {}

    def correr():
        try:
            pm.execute_notebook(
                str(entrada), str(salida),
                parameters=parametros, kernel_name="python3",
                progress_bar=False, log_output=False,
            )
        except BaseException as error:
            resultado["fallo"] = error

    hilo = threading.Thread(target=correr, daemon=True)
    hilo.start()

    while True:
        hilo.join(timeout=LATIDO_SEGUNDOS)
        if not hilo.is_alive():
            break
        hechas = etapas.estado(nombre)["unidades_procesadas"]
        detalle = f"{hechas} unidades procesadas" if hechas else "preparando el entorno"
        print(f"    ... {(time.time() - inicio) / 60:5.1f} min · {detalle}")

    fallo = resultado.get("fallo")

    # nbclient instala manejadores de señales, que sólo funcionan en el hilo principal. Si el
    # entorno lo rechaza, se reintenta de forma bloqueante: se pierde el latido, no la corrida.
    if isinstance(fallo, ValueError) and "signal only works in main thread" in str(fallo):
        print("    (sin latido: este entorno no permite correr papermill en un hilo)")
        try:
            pm.execute_notebook(
                str(entrada), str(salida),
                parameters=parametros, kernel_name="python3",
                progress_bar=False, log_output=False,
            )
            return None
        except Exception as error:
            return error

    return fallo


resumen = []
orden = etapas.orden_de_ejecucion()

print(f"Notebooks en : {CODIGO}")
print(f"Artefactos en: {rutas.raiz()}")
print(f"Etapas       : {len(orden)} -> {', '.join(orden)}\n")

for nombre in orden:
    etapa = etapas.PIPELINE[nombre]
    forzar = nombre in ETAPAS_A_FORZAR
    situacion = etapas.estado(nombre)["estado"]

    if situacion == etapas.COMPLETA and not forzar:
        print(f"[=] {nombre:<20} ya completa, se saltea")
        resumen.append({"etapa": nombre, "resultado": "salteada", "segundos": 0})
        continue

    if forzar:
        print(f"[!] {nombre:<20} re-ejecución forzada: se descartan sus artefactos")
        etapas.limpiar(nombre)

    entrada = CODIGO / etapa.notebook
    if not entrada.exists():
        raise SystemExit(f"No se encontró {entrada}. Poné SINCRONIZAR_CODIGO = True y reejecutá el Paso 0.")

    marca = datetime.now().strftime("%Y%m%d_%H%M%S")
    # Papermill escribe la salida celda por celda mientras ejecuta. Se escribe al disco local y
    # se copia a Drive al final: escribir en Drive durante una hora de ejecución es lento y
    # más frágil que hacerlo una sola vez.
    salida_local = rutas.dir_trabajo("ejecuciones") / f"{nombre}_{marca}.ipynb"

    print(f"[>] {nombre:<20} ejecutando {etapa.notebook} ({situacion})")
    print(f"    la instalación de dependencias tarda varios minutos y no imprime nada")
    inicio = time.time()

    # EJECUCION_AUTOMATICA apaga las barras de progreso y la salida verbosa de los hijos:
    # bajo papermill esa salida se acumula en memoria hasta agotar la RAM de la sesión.
    parametros = {"EJECUCION_AUTOMATICA": True, **PARAMETROS.get(nombre, {})}

    fallo = ejecutar_etapa(nombre, entrada, salida_local, parametros, inicio)

    transcurrido = time.time() - inicio
    sufijo = "_FALLIDA" if fallo else ""
    destino = rutas.carpeta("ejecuciones") / f"{nombre}_{marca}{sufijo}.ipynb"
    if salida_local.exists():
        shutil.copy(salida_local, destino)

    if fallo:
        print(f"\n[x] {nombre:<20} FALLÓ tras {transcurrido / 60:.1f} min")
        print(f"    {type(fallo).__name__}: {fallo}")
        print(f"    copia con el error: {destino}")
        resumen.append({"etapa": nombre, "resultado": "fallida", "segundos": transcurrido})
        dependientes = [n for n, e in etapas.PIPELINE.items() if nombre in e.depende_de]
        if dependientes:
            print(f"\nSe detiene el pipeline: {', '.join(dependientes)} dependen de esta etapa.")
        break

    print(f"[v] {nombre:<20} OK en {transcurrido / 60:.1f} min -> {destino}")
    resumen.append({"etapa": nombre, "resultado": "ejecutada", "segundos": transcurrido})


## Paso 4 — Resumen de la corrida

Si el kernel se reinició, el resumen en memoria se pierde pero el estado real no: está en los
artefactos de Drive, y el diagnóstico del final lo lee de ahí.


In [ ]:
# ============================================================
# RESUMEN DE LA CORRIDA
# ============================================================
# Deliberadamente sin pandas ni numpy.
#
# Los notebooks de etapa corren en kernels separados pero comparten el site-packages de la
# sesión, y algunos instalan sus propias dependencias: DeepLabCut fija su versión de numpy.
# Eso puede dejar el stack científico de ESTE kernel inconsistente con lo que ya tenía cargado,
# y `import pandas` falla con "numpy.dtype size changed".
#
# El orquestador no debe depender de lo que sus hijos mutan. Con la biblioteca estándar alcanza
# de sobra para imprimir una tabla.

if "resumen" in globals() and resumen:
    ancho = max(len(f["etapa"]) for f in resumen)

    print(f"{'ETAPA'.ljust(ancho)}  {'RESULTADO':<10}  MINUTOS")
    print("-" * (ancho + 22))
    for fila in resumen:
        print(f"{fila['etapa'].ljust(ancho)}  {fila['resultado']:<10}  {fila['segundos'] / 60:7.1f}")

    print(f"\nTiempo total: {sum(f['segundos'] for f in resumen) / 60:.1f} min")
else:
    print("Sin resumen en memoria: el kernel se reinició o el Paso 3 no llegó a correr.")
    print("El estado real de cada etapa está abajo, leído de los artefactos en Drive.")

print("\nEstado final del pipeline:\n")
etapas.diagnostico()
